# Replication on judge labels: can a null fit on correct answers flag confident hallucinations?

EE 456 honors option, Nov 6 checkpoint. Dr. Miller's Oct 17 point: a supervised probe needs a representative set of hallucinations, which nobody has in practice. So treat detection as **unsupervised anomaly detection**: learn what *correct* answers look like inside the model, and ask whether confident hallucinations look atypical.

**The procedure, as a prediction interval.** In the battery problem, x̄ and s come from good batteries, and a new battery outside x̄ ± t·s·√(1+1/n) is suspected defective. Here:

| Battery problem | This notebook |
|---|---|
| Good batteries | Correctly answered questions (judge labels) |
| Lifetime | Anomaly score of the residual-stream activation (last prompt token, one layer) |
| x̄, s | Mean and spread of correct-answer activations (the *fit* set) |
| t·s·√(1+1/n) from a t-table | 95th percentile of scores of *other* correct answers (the *calibration* set), no normality needed |
| Battery outside the interval | p ≤ 0.05: flagged as a possible hallucination |

p-value of a new answer = (1 + number of calibration scores at least as large) / (n_cal + 1). Three anomaly scores: **Mahalanobis** (distance inside the top-20 PCA directions of correct answers, in units of their spread), **PCA residual** (how much of the activation sticks out of those directions), **kNN** (mean distance to the 10 nearest correct answers; assumes no shape). Code and tests: `src/pilot/anomaly.py`, `tests/test_anomaly.py`.

**Two checks.** (1) *Calibration*: about 5% of held-out correct answers should be flagged. (2) *The result*: what share of confident hallucinations is flagged. The supervised probe from `lit_review.ipynb` (judge labels) is the ceiling: it is allowed to see hallucinations.

**Cross-fitting.** Correct answers are split into 5 folds. Each fold is scored by a null fit and calibrated on the other 4 folds (2/3 fit, 1/3 calibration), so every answer gets exactly one p-value from a model that never saw it. Hallucinations are spread over the same folds.

In [1]:
import json, sys
from pathlib import Path
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
from sklearn.metrics import roc_auc_score

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "src" / "pilot"))
from anomaly import SCORE_NAMES, crossfit_pvalues, fit_null, score

FIG_DIR = ROOT / "results" / "figs_replication"; FIG_DIR.mkdir(parents=True, exist_ok=True)
INK, MUTED, RULE, PAPER, ACCENT = "#26251f", "#6f6f69", "#c9c6bd", "#eeece6", "#bd6593"
SCORE_COLORS = {"mahalanobis": "#4a7c8c", "pca_residual": "#b08a3c", "knn": MUTED, "probe": ACCENT}
SCORE_LABELS = {"mahalanobis": "Mahalanobis", "pca_residual": "PCA residual", "knn": "kNN", "probe": "supervised probe (ceiling)"}
plt.rcParams.update({
    "figure.facecolor": PAPER, "axes.facecolor": PAPER, "savefig.facecolor": PAPER,
    "axes.edgecolor": MUTED, "axes.labelcolor": INK, "xtick.color": MUTED, "ytick.color": MUTED,
    "text.color": INK, "axes.spines.top": False, "axes.spines.right": False,
    "axes.grid": True, "grid.color": RULE, "grid.linewidth": 0.6, "axes.axisbelow": True,
    "font.family": "sans-serif", "font.size": 10, "axes.titlesize": 12, "axes.titlelocation": "left",
    "legend.frameon": False, "figure.dpi": 110,
})

THRESHOLD, ALPHA = 0.8, 0.05
N_COMPONENTS, KNN_K, N_FOLDS, SEED = 20, 10, 5, 0
MIN_POOL = 200  # smallest null pool we fit (fit set ~ MIN_POOL * 4/5 * 2/3 = 107 points for 20 components)
RUNS = {"Qwen3-0.6B": "pilot_qwen3_0.6b", "Qwen3-4B": "pilot_qwen3_4b"}
PROBE_LAYER = {"Qwen3-0.6B": 18, "Qwen3-4B": 24}  # best layers from the supervised sweep (lit_review, judge labels)

def load_run(run):
    d = pd.read_json(ROOT / "results" / run / "records.jsonl", lines=True)
    d["row"] = np.arange(len(d))
    j = pd.read_json(ROOT / "results" / run / "judge_labels.jsonl", lines=True)
    verdict = j.assign(key=j.qid + "\t" + j.answer.str.strip().str.lower()).drop_duplicates("key").set_index("key").verdict
    d["verdict"] = (d.qid + "\t" + d.answer.str.strip().str.lower()).map(verdict)
    if d.verdict.isna().any(): raise ValueError(f"{run}: answers without a judge verdict")
    d = d[d.verdict != "ambiguous"].reset_index(drop=True)
    d["correct"] = (d.verdict == "correct").astype(int)
    d["confident"] = d.first_tok_prob >= THRESHOLD
    # semantic entropy: first 500 questions ("explore", used to pick layers) + questions 501-2000 ("holdout", if run)
    parts = [pd.read_json(p, lines=True)[["qid", "semantic_entropy"]].assign(se_split=split)
             for name, split in [("semantic_entropy.jsonl", "explore"), ("semantic_entropy_holdout.jsonl", "holdout")]
             if (p := ROOT / "results" / run / name).exists()]
    se = pd.concat(parts, ignore_index=True)
    if not se.qid.is_unique: raise ValueError(f"{run}: a question appears in both semantic-entropy files")
    d = d.merge(se, on="qid", how="left")
    d["hard"] = d.confident & (d.semantic_entropy == 0)  # confident and stable under resampling
    d["hard_explore"] = d.hard & (d.se_split == "explore")
    d["hard_holdout"] = d.hard & (d.se_split == "holdout")
    return d

def load_acts(run, rows):
    return np.load(ROOT / "data" / "activations" / run / "last_prompt_token.npy", mmap_mode="r")[rows]

REC = {m: load_run(r) for m, r in RUNS.items()}
PROBE = {m: np.load(ROOT / "results" / r / "oof_scores_judge.npz")["probe_last_prompt"] for m, r in RUNS.items()}
POOLS = {m: {name: sel for name, sel in [("all correct", d.correct == 1), ("confident correct", (d.correct == 1) & d.confident)]
             if sel.sum() >= MIN_POOL} for m, d in REC.items()}
pd.DataFrame({m: {"answers": len(d), "correct": int(d.correct.sum()), "confident correct": int((d.correct & d.confident).sum()),
                  "confident wrong": int(((d.correct == 0) & d.confident).sum()),
                  "questions with semantic entropy": int(d.semantic_entropy.notna().sum()),
                  "hard set (wrong / total)": f"{int(((d.correct == 0) & d.hard).sum())} / {int(d.hard.sum())}",
                  "  first 500 (explore)": f"{int(((d.correct == 0) & d.hard_explore).sum())} / {int(d.hard_explore.sum())}",
                  "  501-2000 (holdout)": f"{int(((d.correct == 0) & d.hard_holdout).sum())} / {int(d.hard_holdout.sum())}",
                  "null pools fit": ", ".join(POOLS[m])} for m, d in REC.items()}).T

,answers,correct,confident correct,confident wrong,questions with semantic entropy,hard set (wrong / total),first 500 (explore),501-2000 (holdout),null pools fit
Qwen3-0.6B,1983,253,89,544,1983,63 / 81,12 / 14,51 / 67,all correct
Qwen3-4B,1989,831,681,367,1989,174 / 773,47 / 171,127 / 602,"all correct, confident correct"


## 1. p-values at every layer

For each model, null pool, and layer, every answer gets one cross-fit p-value per score. Cached in `results/<run>/anomaly_pvals_judge.npz`.

In [2]:
FORCE_RECOMPUTE = False

def pvalues_by_layer(model, run):
    cache = ROOT / "results" / run / "anomaly_pvals_judge.npz"
    # signature: the exact rows scored and every setting that changes a p-value; a mismatch forces a recompute
    sig = np.array(json.dumps({"qids": list(REC[model].qid), "pools": {k: list(map(bool, v)) for k, v in POOLS[model].items()},
                               "params": [N_COMPONENTS, KNN_K, N_FOLDS, SEED]}))
    if cache.exists() and not FORCE_RECOMPUTE:
        z = np.load(cache)
        if "signature" in z.files and str(z["signature"]) == str(sig):
            return {k: z[k] for k in z.files if k != "signature"}
        print(f"{model}: cache is stale, recomputing", flush=True)
    acts = load_acts(run, REC[model].row.values)
    n_layers = acts.shape[1]; out = {}
    for pool, sel in POOLS[model].items():
        per_layer = [crossfit_pvalues(acts[:, l, :].astype(np.float32), sel.values, N_COMPONENTS, KNN_K, N_FOLDS, SEED)
                     for l in range(n_layers)]
        for name in SCORE_NAMES:
            out[f"{pool}|{name}"] = np.stack([p[name] for p in per_layer])
        print(f"{model} / {pool}: {n_layers} layers done", flush=True)
    np.savez_compressed(cache, signature=sig, **out); return out

PV = {m: pvalues_by_layer(m, r) for m, r in RUNS.items()}
{m: sorted(v) for m, v in PV.items()}

{'Qwen3-0.6B': ['all correct|knn',
  'all correct|mahalanobis',
  'all correct|pca_residual'],
 'Qwen3-4B': ['all correct|knn',
  'all correct|mahalanobis',
  'all correct|pca_residual',
  'confident correct|knn',
  'confident correct|mahalanobis',
  'confident correct|pca_residual']}

## 2. AUROC by layer: anomaly scores vs the supervised probe

AUROC of the p-value for telling correct from wrong (higher p = more typical = predicted correct), within each subset. The probe line is the ceiling. Null pool: all correct answers.

In [3]:
def auroc(y, s):
    y = np.asarray(y); return float(roc_auc_score(y, s)) if len(np.unique(y)) == 2 else float("nan")

MIN_CLASS = 20  # a subset is evaluated only if it has at least this many correct and this many wrong answers
def evaluable(d, sel):
    return min(int((d.correct[sel] == 1).sum()), int((d.correct[sel] == 0).sum())) >= MIN_CLASS
SUBSETS = {m: [s for s in ("confident", "hard") if evaluable(d, d[s])] for m, d in REC.items()}
SUBSET_LABELS = {"confident": "confident answers", "hard": "hard set (confident & stable)"}

rows = []
for m, d in REC.items():
    for sub in SUBSETS[m]:
        sel = d[sub].values; y = d.correct.values[sel]
        for pool in POOLS[m]:
            for name in SCORE_NAMES:
                for l, p in enumerate(PV[m][f"{pool}|{name}"]):
                    rows.append({"model": m, "subset": sub, "pool": pool, "score": name, "layer": l, "AUROC": auroc(y, p[sel])})
        for l, s in enumerate(PROBE[m]):
            rows.append({"model": m, "subset": sub, "pool": "-", "score": "probe", "layer": l, "AUROC": auroc(y, s[sel])})
BY_LAYER = pd.DataFrame(rows)

panels = [(m, sub) for m in REC for sub in SUBSETS[m]]
fig, axes = plt.subplots(1, len(panels), figsize=(5.2 * len(panels), 4.2), sharey=True)
for ax, (m, sub) in zip(axes, panels):
    g = BY_LAYER[(BY_LAYER.model == m) & (BY_LAYER.subset == sub)]
    for name in ("probe", *SCORE_NAMES):
        h = g[(g.score == name) & (g.pool.isin(["-", "all correct"]))]
        ax.plot(h.layer, h.AUROC, color=SCORE_COLORS[name], lw=2.2 if name == "probe" else 1.6, label=SCORE_LABELS[name])
    ax.axhline(0.5, color=MUTED, lw=0.8, ls="--"); ax.axvline(PROBE_LAYER[m], color=RULE, lw=1)
    n = int(REC[m][sub].sum()); ax.set_title(f"{m}: {SUBSET_LABELS[sub]} (n={n})", fontsize=10.5); ax.set_xlabel("layer")
axes[0].set_ylabel("AUROC (correct vs wrong)"); axes[0].legend(fontsize=8.5, loc="lower right")
fig.tight_layout(); fig.savefig(FIG_DIR / "01_auroc_by_layer.png", dpi=160, bbox_inches="tight"); plt.show()

best = BY_LAYER[BY_LAYER.pool.isin(["-", "all correct"])].loc[lambda t: t.groupby(["model", "subset", "score"]).AUROC.idxmax()]
best.pivot_table(index=["model", "subset"], columns="score", values=["AUROC", "layer"]).round(3)

/var/folders/1f/rj7xg3297314237bcmb784h00000gn/T/ipykernel_85447/2110163169.py:32: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  fig.tight_layout(); fig.savefig(FIG_DIR / "01_auroc_by_layer.png", dpi=160, bbox_inches="tight"); plt.show()


AUROC                                 layer              \
score                   knn mahalanobis pca_residual  probe   knn mahalanobis   
model      subset                                                               
Qwen3-0.6B confident  0.708       0.673        0.691  0.805   1.0         1.0   
Qwen3-4B   confident  0.644       0.593        0.627  0.801  36.0        19.0   
           hard       0.686       0.607        0.670  0.798  21.0        19.0   

                                         
score                pca_residual probe  
model      subset                        
Qwen3-0.6B confident          1.0  18.0  
Qwen3-4B   confident         21.0  24.0  
           hard              21.0  24.0

**Read:** the pink line is the ceiling. At 4B every anomaly score sits well below it at every layer. On the hard set, PCA residual and kNN climb slowly to a peak of about 0.67-0.69 at layers 20-21, then drop *below* 0.5 in the last layers, where hallucinations look *more* typical than correct answers. Mahalanobis stays near 0.5-0.6 throughout. At 0.6B the anomaly scores peak at layer 1 and decline from there. That early, the last-token state mostly encodes what the question looks like, so a layer-1 signal more likely reflects question type or difficulty than a hallucination state. Each score's best layer in the table below was picked with the labels; Section 3b tests whether the 4B peak survives on fresh questions.

## 3. The prediction-interval result at the probe's layer

Layer fixed in advance at the supervised probe's best layer (18 for 0.6B, 24 for 4B), so no anomaly number is tuned on hallucination labels. For each score: AUROC with bootstrap CI, the share of correct answers flagged at p ≤ 0.05 (calibration check, should be about 5%), the share of wrong answers flagged (the result), and a one-sided Fisher exact test of whether wrong answers are flagged more often than correct answers in the same subset, with a Holm correction across all rows.

Caveats: the p-values are descriptive. Answers in one fold share a calibration set, so their flags are not fully independent. The probe row's threshold is set on the subset's own correct answers, so it flags exactly 5% by construction. The anomaly cutoff comes from separate calibration answers and can land slightly under 5%. The comparison therefore slightly favours the probe.

In [4]:
def boot_ci(y, s, n=1000, seed=0):
    rng = np.random.default_rng(seed); vals = []
    for _ in range(n):
        i = rng.integers(0, len(y), len(y))
        if len(np.unique(y[i])) == 2: vals.append(roc_auc_score(y[i], s[i]))
    return np.percentile(vals, [2.5, 97.5])

def holm(p):
    """Holm step-down adjusted p-values: sorted p_(j) times (m - j), running max, capped at 1."""
    order = np.argsort(p); m = len(p)
    adjusted = np.minimum(np.maximum.accumulate((m - np.arange(m)) * p[order]), 1.0)
    out = np.empty(m); out[order] = adjusted; return out

def flag_row(y, flagged):
    k_c, n_c = int(flagged[y == 1].sum()), int((y == 1).sum()); k_w, n_w = int(flagged[y == 0].sum()), int((y == 0).sum())
    lo, hi = stats.binomtest(k_w, n_w).proportion_ci(method="wilson")  # two-sided 95% CI
    fisher_p = stats.fisher_exact([[k_w, n_w - k_w], [k_c, n_c - k_c]], alternative="greater").pvalue
    return {"correct flagged": f"{k_c}/{n_c} ({k_c / n_c:.0%})", "wrong flagged": f"{k_w}/{n_w} ({k_w / n_w:.0%})",
            "wrong flagged CI": f"{lo:.0%}-{hi:.0%}", "Fisher p": fisher_p}

rows = []; FLAGS = {}
for m, d in REC.items():
    l = PROBE_LAYER[m]
    for sub in SUBSETS[m]:
        sel = d[sub].values; y = d.correct.values[sel]
        for pool in POOLS[m]:
            for name in SCORE_NAMES:
                p = PV[m][f"{pool}|{name}"][l][sel]; lo, hi = boot_ci(y, p)
                FLAGS[(m, sub, pool, name)] = p <= ALPHA
                rows.append({"model": m, "subset": sub, "pool": pool, "score": name, "AUROC": auroc(y, p),
                             "CI": f"{lo:.2f}-{hi:.2f}", **flag_row(y, p <= ALPHA)})
        s = PROBE[m][l][sel]; lo, hi = boot_ci(y, s)
        flagged = s <= np.quantile(s[y == 1], ALPHA)  # low probe score = predicted wrong
        FLAGS[(m, sub, "-", "probe")] = flagged
        rows.append({"model": m, "subset": sub, "pool": "-", "score": "probe", "AUROC": auroc(y, s),
                     "CI": f"{lo:.2f}-{hi:.2f}", **flag_row(y, flagged)})
RESULT = pd.DataFrame(rows)
RESULT["Holm p"] = holm(RESULT["Fisher p"].values)
RESULT.style.format({"AUROC": "{:.3f}", "Fisher p": "{:.1e}", "Holm p": "{:.1e}"}).hide(axis="index")

model,subset,pool,score,AUROC,CI,correct flagged,wrong flagged,wrong flagged CI,Fisher p,Holm p
Qwen3-0.6B,confident,all correct,mahalanobis,0.484,0.42-0.55,5/89 (6%),22/544 (4%),3%-6%,8.3e-01,1.0e+00
Qwen3-0.6B,confident,all correct,pca_residual,0.563,0.50-0.63,1/89 (1%),14/544 (3%),2%-4%,3.5e-01,1.0e+00
Qwen3-0.6B,confident,all correct,knn,0.526,0.46-0.60,2/89 (2%),13/544 (2%),1%-4%,6.5e-01,1.0e+00
Qwen3-0.6B,confident,-,probe,0.805,0.76-0.86,5/89 (6%),164/544 (30%),26%-34%,6.3e-08,1.0e-06
Qwen3-4B,confident,all correct,mahalanobis,0.508,0.47-0.54,29/681 (4%),9/367 (2%),1%-5%,9.6e-01,1.0e+00
Qwen3-4B,confident,all correct,pca_residual,0.553,0.51-0.59,36/681 (5%),30/367 (8%),6%-11%,4.6e-02,5.5e-01
Qwen3-4B,confident,all correct,knn,0.577,0.54-0.61,37/681 (5%),28/367 (8%),5%-11%,1.0e-01,9.2e-01
Qwen3-4B,confident,confident correct,mahalanobis,0.513,0.48-0.55,33/681 (5%),10/367 (3%),1%-5%,9.7e-01,1.0e+00
Qwen3-4B,confident,confident correct,pca_residual,0.573,0.54-0.61,37/681 (5%),28/367 (8%),5%-11%,1.0e-01,9.2e-01
Qwen3-4B,confident,confident correct,knn,0.596,0.56-0.63,31/681 (5%),33/367 (9%),6%-12%,3.7e-03,5.6e-02


**Read:** calibration holds: 1-6% of correct answers are flagged in every row, as designed.

At the probe's layer the anomaly null is weak. On the 4B hard set (174 hallucinations, now that semantic entropy covers all 2,000 questions), PCA residual and kNN flag 9-10% of hallucinations against 4-5% of correct answers. The best AUROC is 0.62, Mahalanobis is at chance, and no anomaly row survives the Holm correction (closest: kNN with the confident-correct null, Holm p = 0.09). At 0.6B no anomaly score beats chance at layer 18.

The probe flags 28-31% at the same false-alarm rate, and every probe row is significant.

## 3b. Is the best layer real? Choose on the first 500 questions, test on the other 1,500

The layer sweep in Section 2 picks each score's best layer by looking at hallucination labels, which flatters the winner. Here the layer is chosen only on the hard set from the first 500 questions (where semantic entropy was first run), and then scored once on the hard set from questions 501-2000, which played no part in the choice. If the held-out AUROC stays near the chosen one, the layer effect is real; if it falls toward the fixed-layer numbers, it was luck. The probe is treated the same way; its scores are still out-of-fold.

In [5]:
rows = []
for m, d in REC.items():
    ex, ho = d.hard_explore.values, d.hard_holdout.values
    if not (evaluable(d, d.hard_explore) and evaluable(d, d.hard_holdout)):
        print(f"{m}: hard set too small to split (needs {MIN_CLASS} correct and {MIN_CLASS} wrong in each half)"); continue
    y_ex, y_ho = d.correct.values[ex], d.correct.values[ho]
    candidates = [(pool, name, PV[m][f"{pool}|{name}"]) for pool in POOLS[m] for name in SCORE_NAMES] + [("-", "probe", PROBE[m])]
    for pool, name, S in candidates:
        auc_ex = np.array([auroc(y_ex, s[ex]) for s in S]); l_best = int(np.nanargmax(auc_ex))
        s_ho = S[l_best][ho]; lo, hi = boot_ci(y_ho, s_ho)
        flagged = s_ho <= ALPHA if name != "probe" else s_ho <= np.quantile(s_ho[y_ho == 1], ALPHA)
        rows.append({"model": m, "pool": pool, "score": name, "layer chosen": l_best, "AUROC first 500 (chosen)": auc_ex[l_best],
                     "AUROC held-out": auroc(y_ho, s_ho), "held-out CI": f"{lo:.2f}-{hi:.2f}",
                     "held-out AUROC at probe layer": auroc(y_ho, S[PROBE_LAYER[m]][ho]), **flag_row(y_ho, flagged)})
CONFIRM = pd.DataFrame(rows)
if len(CONFIRM):
    CONFIRM["Holm p"] = holm(CONFIRM["Fisher p"].values)
    display(CONFIRM.style.format({c: "{:.3f}" for c in ["AUROC first 500 (chosen)", "AUROC held-out", "held-out AUROC at probe layer"]}
                                 | {"Fisher p": "{:.1e}", "Holm p": "{:.1e}"}).hide(axis="index"))

Qwen3-0.6B: hard set too small to split (needs 20 correct and 20 wrong in each half)


model,pool,score,layer chosen,AUROC first 500 (chosen),AUROC held-out,held-out CI,held-out AUROC at probe layer,correct flagged,wrong flagged,wrong flagged CI,Fisher p,Holm p
Qwen3-4B,all correct,mahalanobis,20,0.656,0.528,0.47-0.59,0.484,18/475 (4%),12/127 (9%),5%-16%,1.2e-02,3.7e-02
Qwen3-4B,all correct,pca_residual,20,0.733,0.631,0.58-0.68,0.575,22/475 (5%),14/127 (11%),7%-18%,9.2e-03,3.7e-02
Qwen3-4B,all correct,knn,20,0.730,0.653,0.60-0.70,0.596,26/475 (5%),18/127 (14%),9%-21%,1.5e-03,7.7e-03
Qwen3-4B,confident correct,mahalanobis,20,0.671,0.534,0.48-0.60,0.484,16/475 (3%),11/127 (9%),5%-15%,1.4e-02,3.7e-02
Qwen3-4B,confident correct,pca_residual,20,0.735,0.645,0.59-0.70,0.596,17/475 (4%),10/127 (8%),4%-14%,3.9e-02,3.9e-02
Qwen3-4B,confident correct,knn,21,0.722,0.682,0.63-0.73,0.618,22/475 (5%),18/127 (14%),9%-21%,3.7e-04,2.2e-03
Qwen3-4B,-,probe,24,0.764,0.811,0.77-0.85,0.811,24/475 (5%),39/127 (31%),23%-39%,6.1e-14,4.3e-13


**Read:** the layer-20 effect is real but smaller than the sweep suggested. The layer was chosen on the first 500 questions, where kNN and PCA residual reached about 0.73. On the 1,500 held-out questions (127 hard-set hallucinations) they score 0.63-0.68, with CIs clear of 0.5. They flag 11-14% of hallucinations against about 5% of correct answers, and this survives the Holm correction (best: kNN, Holm p = 0.002-0.008). Mahalanobis falls to about 0.53.

So part of the 0.73 was selection luck. A modest unsupervised signal remains at layers 20-21, about 0.05-0.07 above what the same scores reach at the probe's layer. The probe, at its own pre-chosen layer, reaches 0.81 on the same held-out questions. At 0.6B the hard set (63 wrong, 18 correct) is too small to split.

## 4. Is the normal shape a fair assumption? (Mahalanobis check)

If correct-answer activations were Gaussian, the Mahalanobis scores of the *fit* points would follow a χ² distribution with 20 degrees of freedom, so their quantiles would sit on the diagonal. The second line is held-out correct answers scored by the same fit. If in-sample scores were a fair guide to new answers, the two lines would overlap. The title compares the 95% cutoff a χ² table would give with the one read off held-out correct answers.

In [6]:
fig, axes = plt.subplots(1, len(REC), figsize=(5.6 * len(REC), 4.4))
for ax, (m, d) in zip(axes, REC.items()):
    idx = np.flatnonzero(d.correct.values == 1); rng = np.random.default_rng(SEED); rng.shuffle(idx)
    half = len(idx) // 2
    x = load_acts(RUNS[m], d.row.values[idx])[:, PROBE_LAYER[m], :].astype(np.float32)
    null = fit_null(x[:half], N_COMPONENTS, KNN_K)
    q = np.linspace(0.01, 0.99, 99); ref = stats.chi2.ppf(q, N_COMPONENTS)
    for label, pts, c in [("fit points (in-sample)", x[:half], INK), ("held-out correct answers", x[half:], ACCENT)]:
        ax.plot(ref, np.quantile(score(null, pts)["mahalanobis"], q), "o", ms=3, color=c, label=label)
    lim = max(ref.max(), np.quantile(score(null, x[half:])["mahalanobis"], 0.99)); ax.plot([0, lim], [0, lim], color=MUTED, lw=0.8, ls="--")
    cut_chi2, cut_emp = stats.chi2.ppf(1 - ALPHA, N_COMPONENTS), np.quantile(score(null, x[half:])["mahalanobis"], 1 - ALPHA)
    ax.set_title(f"{m}, layer {PROBE_LAYER[m]}: 95% cutoff, χ² table {cut_chi2:.0f} vs data {cut_emp:.0f}", fontsize=10)
    ax.set_xlabel(f"χ²({N_COMPONENTS}) quantile (what a Gaussian predicts)"); ax.set_ylabel("Mahalanobis quantile (observed)")
axes[0].legend(fontsize=9)
fig.tight_layout(); fig.savefig(FIG_DIR / "02_mahalanobis_qq.png", dpi=160, bbox_inches="tight"); plt.show()

/var/folders/1f/rj7xg3297314237bcmb784h00000gn/T/ipykernel_85447/2496089416.py:15: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  fig.tight_layout(); fig.savefig(FIG_DIR / "02_mahalanobis_qq.png", dpi=160, bbox_inches="tight"); plt.show()


**Read:** two departures from the Gaussian picture.

1. The fit points bend upward at the top. The tails are heavier than a Gaussian's, so a χ² table would misplace the cutoff.
2. The held-out points sit *below* the fit points, the opposite of the one-dimensional battery case. With about 125 (0.6B) to 415 (4B) fit points in 1,024 to 2,560 dimensions, the top PCA directions partly describe the fit set's own noise. Fit points look more spread out along those directions than new points do.

In either direction, in-sample scores misrepresent new answers. That is why the cutoff comes from a separate calibration set rather than from the fit points or a table.

## 5. Case by case on the 4B hard set

The hard-set hallucinations (confident, same answer on every resample, wrong). Which ones does each method flag at a 5% false-alarm rate, and does the anomaly null catch any the probe misses?

In [7]:
m, sub, pool = "Qwen3-4B", "hard", "all correct"
d = REC[m]; sel = d[sub].values; wrong = d.correct.values[sel] == 0
methods = {SCORE_LABELS[n]: FLAGS[(m, sub, pool, n)][wrong] for n in SCORE_NAMES}
methods[SCORE_LABELS["probe"]] = FLAGS[(m, sub, "-", "probe")][wrong]
any_anomaly = np.any([FLAGS[(m, sub, pool, n)][wrong] for n in SCORE_NAMES], axis=0)
probe_f = methods[SCORE_LABELS["probe"]]
summary = pd.Series({**{k: int(v.sum()) for k, v in methods.items()},
                     "any anomaly score": int(any_anomaly.sum()),
                     "probe and an anomaly score": int((probe_f & any_anomaly).sum()),
                     "anomaly only (probe misses)": int((~probe_f & any_anomaly).sum()),
                     "probe only": int((probe_f & ~any_anomaly).sum()),
                     "neither": int((~probe_f & ~any_anomaly).sum())}, name=f"of {int(wrong.sum())} hard-set hallucinations")
summary.to_frame()

,of 174 hard-set hallucinations
Mahalanobis,7
PCA residual,17
kNN,15
supervised probe (ceiling),49
any anomaly score,23
probe and an anomaly score,8
anomaly only (probe misses),15
probe only,41
neither,110


**Read:** at the probe's layer (24), together the anomaly scores flag 23 of the 174 hard-set hallucinations, and 15 of those are ones the probe misses. The probe flags 49. Most (110) are flagged by nothing at a 5% false-alarm rate. The two kinds of signal overlap little: the probe finds a direction that separates hallucinations, and the anomaly scores pick up a different, smaller group of unusual-looking cases.